# Unit 5 — 07: A/B Testing and Canary Deployment for Model Releases

**What you will do:** Build an `ABRouter` that splits traffic between two models, analyze the results with statistical tests, then build a `CanaryDeployment` class that gradually promotes the new model — or rolls it back.

**Why it matters:** Replacing a production model immediately is high-risk. A/B testing and canary deployments let you measure the real impact on live traffic before committing.

**How to run:** Python 3.10+. Run cells in order.

---

## Section 1 — Why Not Replace Model A with Model B Immediately?

A new model may score higher on your held-out test set but still:
- Perform worse on specific user segments not well-represented in test data
- Have higher latency under production load
- Produce confident but wrong predictions on edge cases

**A/B testing** routes a fraction of live traffic to the new model and measures real outcomes side by side.  
**Canary deployment** makes the rollout gradual: 5% → 25% → 100%, with automatic rollback if metrics degrade at any step.

## Section 2 — A/B Testing Setup

In [ ]:
import numpy as np
import time
from typing import List, Dict, Tuple

import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(42)

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Model A: current production model. We deliberately under-regularize it
# (small C) so it is a genuinely WEAKER baseline (~0.82 accuracy) — this gives
# the A/B test a real difference to detect, the way an older model would.
model_a = LogisticRegression(C=0.008, max_iter=200, random_state=42)
model_a.fit(X_train, y_train)

# Model B: candidate model (RandomForest — should be better)
model_b = RandomForestClassifier(n_estimators=50, random_state=42)
model_b.fit(X_train, y_train)

print(f"Model A (LogisticRegression) test accuracy: {model_a.score(X_test, y_test):.4f}")
print(f"Model B (RandomForest)        test accuracy: {model_b.score(X_test, y_test):.4f}")


class ABRouter:
    """Routes a fraction of traffic to model B; the rest goes to model A."""

    def __init__(self, model_a: object, model_b: object, b_fraction: float = 0.10):
        self.model_a = model_a
        self.model_b = model_b
        self.b_fraction = b_fraction
        self._rng = np.random.default_rng(0)

    def predict(self, X: np.ndarray) -> Tuple[int, str, float]:
        """
        Returns (prediction, model_used, confidence).
        model_used is 'A' or 'B'.
        """
        use_b = self._rng.random() < self.b_fraction
        model = self.model_b if use_b else self.model_a
        proba = model.predict_proba(X.reshape(1, -1))[0]
        prediction = int(np.argmax(proba))
        confidence = float(np.max(proba))
        model_used = "B" if use_b else "A"
        return prediction, model_used, confidence


router = ABRouter(model_a, model_b, b_fraction=0.50)
print(f"\nABRouter created: 50/50 A/B split so both arms get enough samples to compare.")

In [ ]:
# Simulate 1000 requests through the router
N_REQUESTS = 1000
request_log: List[Dict] = []

for i in range(N_REQUESTS):
    idx = rng.integers(0, len(X_test))
    X_sample = X_test[idx]
    true_label = int(y_test[idx])

    t0 = time.perf_counter()
    prediction, model_used, confidence = router.predict(X_sample)
    latency_ms = (time.perf_counter() - t0) * 1000

    request_log.append({
        "model": model_used,
        "prediction": prediction,
        "true_label": true_label,
        "correct": prediction == true_label,
        "confidence": confidence,
        "latency_ms": latency_ms,
    })

a_logs = [r for r in request_log if r["model"] == "A"]
b_logs = [r for r in request_log if r["model"] == "B"]

print(f"Total requests: {N_REQUESTS}")
print(f"  Served by Model A: {len(a_logs)}")
print(f"  Served by Model B: {len(b_logs)}")

## Section 3 — Analyze A/B Results

In [ ]:
import pandas as pd


def summarize_logs(logs: List[Dict], label: str) -> Dict:
    accuracy = np.mean([r["correct"] for r in logs])
    mean_confidence = np.mean([r["confidence"] for r in logs])
    mean_latency = np.mean([r["latency_ms"] for r in logs])
    p95_latency = np.percentile([r["latency_ms"] for r in logs], 95)
    return {
        "Model": label,
        "Requests": len(logs),
        "Accuracy": round(accuracy, 4),
        "Mean Confidence": round(mean_confidence, 4),
        "Mean Latency (ms)": round(mean_latency, 4),
        "p95 Latency (ms)": round(p95_latency, 4),
    }


summary_a = summarize_logs(a_logs, "A (LogisticRegression)")
summary_b = summarize_logs(b_logs, "B (RandomForest)")

df_summary = pd.DataFrame([summary_a, summary_b]).set_index("Model")
print(df_summary.to_string())

# Plot accuracy and confidence
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(["Model A", "Model B"],
            [summary_a["Accuracy"], summary_b["Accuracy"]],
            color=["steelblue", "green"], alpha=0.8)
axes[0].set_ylim(0.8, 1.0)
axes[0].set_title("A/B Test: Accuracy")
axes[0].set_ylabel("Accuracy")

axes[1].bar(["Model A", "Model B"],
            [summary_a["Mean Confidence"], summary_b["Mean Confidence"]],
            color=["steelblue", "green"], alpha=0.8)
axes[1].set_ylim(0.7, 1.0)
axes[1].set_title("A/B Test: Mean Confidence")
axes[1].set_ylabel("Confidence")

plt.tight_layout()
plt.savefig("/tmp/ab_test_results.png", dpi=72)
plt.show()

## Section 4 — Statistical Significance Test

Model B may look better in the A/B data, but is the difference statistically significant or just random variation? Use a two-proportion z-test to check.

In [ ]:
from scipy import stats as sp_stats


def two_proportion_z_test(n_a: int, correct_a: int, n_b: int, correct_b: int) -> Dict:
    """
    Two-proportion z-test: is Model B's accuracy significantly different from Model A's?
    H0: p_B == p_A
    """
    p_a = correct_a / n_a
    p_b = correct_b / n_b
    p_pool = (correct_a + correct_b) / (n_a + n_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    z_stat = (p_b - p_a) / se if se > 0 else 0.0
    # Two-tailed p-value
    p_value = 2 * (1 - sp_stats.norm.cdf(abs(z_stat)))
    return {
        "accuracy_a": round(p_a, 4),
        "accuracy_b": round(p_b, 4),
        "z_statistic": round(z_stat, 4),
        "p_value": round(p_value, 6),
        "significant_at_0.05": p_value < 0.05,
    }


n_a = len(a_logs)
correct_a = sum(r["correct"] for r in a_logs)
n_b = len(b_logs)
correct_b = sum(r["correct"] for r in b_logs)

result = two_proportion_z_test(n_a, correct_a, n_b, correct_b)

print("Two-proportion z-test results:")
for k, v in result.items():
    print(f"  {k}: {v}")

print()
if result["significant_at_0.05"]:
    if result["accuracy_b"] > result["accuracy_a"]:
        print("CONCLUSION: Model B is significantly BETTER than Model A at alpha=0.05.")
    else:
        print("CONCLUSION: Model B is significantly WORSE than Model A at alpha=0.05.")
else:
    print("CONCLUSION: No statistically significant difference detected.")
    print(f"  (Need more data — try n_b >= {int(n_a * 0.5)} before concluding.)")

## Section 5 — Canary Deployment

A canary deployment shifts traffic gradually: 5% → 25% → 100%, checking metrics at each step. If metrics degrade, roll back to 0%.

In [ ]:
class CanaryDeployment:
    """
    Manages a gradual rollout from old_model to new_model.
    fraction = fraction of traffic routed to new_model (0.0 to 1.0).
    """

    STAGES = [0.05, 0.10, 0.25, 0.50, 1.00]

    def __init__(self, old_model: object, new_model: object, initial_fraction: float = 0.05):
        self.old_model = old_model
        self.new_model = new_model
        self.fraction = initial_fraction
        self._stage_idx = 0
        self._rng = np.random.default_rng(7)

    def predict(self, X: np.ndarray) -> Tuple[int, str]:
        use_new = self._rng.random() < self.fraction
        model = self.new_model if use_new else self.old_model
        prediction = int(model.predict(X.reshape(1, -1))[0])
        return prediction, "new" if use_new else "old"

    def promote(self) -> bool:
        """Move to the next traffic stage. Returns True if fully promoted."""
        self._stage_idx += 1
        if self._stage_idx < len(self.STAGES):
            self.fraction = self.STAGES[self._stage_idx]
            return self.fraction == 1.0
        return True

    def rollback(self) -> None:
        """Immediately route all traffic back to old model."""
        self.fraction = 0.0
        self._stage_idx = 0
        print("  ROLLED BACK: 100% traffic back to old model.")

    def status(self) -> str:
        return f"Stage {self._stage_idx}: {self.fraction*100:.0f}% new model traffic"


canary = CanaryDeployment(model_a, model_b, initial_fraction=0.05)
print(f"Canary initialized: {canary.status()}")


def evaluate_canary(
    canary: CanaryDeployment,
    X_eval: np.ndarray,
    y_eval: np.ndarray,
    n_samples: int = 200,
) -> Dict:
    """Run n_samples requests through the canary and return accuracy split."""
    logs = []
    indices = rng.integers(0, len(X_eval), size=n_samples)
    for idx in indices:
        pred, model_used = canary.predict(X_eval[idx])
        logs.append({"model": model_used, "correct": pred == int(y_eval[idx])})

    overall_acc = np.mean([r["correct"] for r in logs])
    new_acc = np.mean([r["correct"] for r in logs if r["model"] == "new"]) if any(
        r["model"] == "new" for r in logs
    ) else 0.0
    return {"overall_accuracy": overall_acc, "new_model_accuracy": new_acc, "n": n_samples}


print("\nCanary promotion simulation:")
print("-" * 60)
stage_results = []

for step in range(5):
    metrics = evaluate_canary(canary, X_test, y_test, n_samples=300)
    overall_acc = metrics["overall_accuracy"]
    status = canary.status()

    print(f"  {status:<35} overall_acc={overall_acc:.3f}")
    stage_results.append({"stage": canary.fraction, "accuracy": overall_acc})

    # Promotion criterion: overall accuracy stays above 0.87
    if overall_acc >= 0.87:
        fully_promoted = canary.promote()
        if fully_promoted:
            print("  FULLY PROMOTED: new model now serves 100% of traffic.")
            break
    else:
        canary.rollback()
        break

In [ ]:
# Visualise the canary stages
stages = [f"{int(s['stage']*100)}%" for s in stage_results]
accuracies = [s["accuracy"] for s in stage_results]

plt.figure(figsize=(8, 4))
plt.plot(stages, accuracies, marker="o", color="green", linewidth=2)
plt.axhline(0.87, color="orange", linestyle="--", label="Promotion threshold (0.87)")
plt.title("Canary Deployment — Accuracy at Each Traffic Stage")
plt.xlabel("% Traffic to New Model")
plt.ylabel("Overall Accuracy")
plt.ylim(0.8, 1.0)
plt.legend()
plt.tight_layout()
plt.savefig("/tmp/canary_stages.png", dpi=72)
plt.show()

## Section 6 — Rollback Criteria

Define explicit, measurable conditions that trigger an automatic rollback.

In [ ]:
def should_rollback(
    old_error_rate: float,
    new_error_rate: float,
    error_rate_delta_threshold: float = 0.005,  # 0.5 percentage points
    new_p99_latency_ms: float = 0.0,
    latency_threshold_ms: float = 500.0,
) -> Tuple[bool, str]:
    """
    Returns (True, reason) if rollback criteria are met, else (False, 'OK').
    """
    if new_error_rate > old_error_rate + error_rate_delta_threshold:
        delta = new_error_rate - old_error_rate
        return True, (
            f"New model error rate {new_error_rate:.3f} exceeds old model "
            f"by {delta:.3f} (threshold={error_rate_delta_threshold})"
        )
    if new_p99_latency_ms > latency_threshold_ms:
        return True, (
            f"New model p99 latency {new_p99_latency_ms:.1f}ms > threshold {latency_threshold_ms:.1f}ms"
        )
    return False, "OK — no rollback needed"


# Scenario 1: New model is slightly worse on error rate
rollback, reason = should_rollback(old_error_rate=0.05, new_error_rate=0.058)
print(f"Scenario 1 — Rollback? {rollback}: {reason}")

# Scenario 2: New model error rate exceeds threshold
rollback, reason = should_rollback(old_error_rate=0.05, new_error_rate=0.062)
print(f"Scenario 2 — Rollback? {rollback}: {reason}")

# Scenario 3: New model is accurate but too slow
rollback, reason = should_rollback(old_error_rate=0.05, new_error_rate=0.04, new_p99_latency_ms=600)
print(f"Scenario 3 — Rollback? {rollback}: {reason}")

## Summary

| Strategy | How it works | When to use |
|---|---|---|
| **A/B test** | Split traffic between A and B; collect metrics; run statistical test | When you want to compare two models with statistical confidence |
| **Canary** | Gradually shift traffic 5% → 100%; roll back on degradation | When you want to limit blast radius of a bad release |

Key principle: **never make promotion automatic without a statistical gate**. The z-test (or chi-squared test) tells you whether observed differences are real or sampling noise.

## Self-Check (answer before scrolling back up)

1. **What sample size do you need before an A/B test is statistically valid?**  
   It depends on the effect size you want to detect and your desired statistical power (typically 80%) and significance level (typically 5%). For a typical ML model with 95% accuracy trying to detect a 1-2% improvement, you usually need several hundred requests per variant minimum. Use a power calculator with your baseline conversion rate and expected delta to get the exact number.

2. **What is the difference between an A/B test and a canary deployment?**  
   An A/B test is primarily a measurement tool — you run both models simultaneously and compare outcomes statistically. A canary deployment is primarily a risk management tool — you shift traffic gradually and roll back if anything goes wrong. They are complementary: you might run an A/B test at the 10% canary stage to validate the decision before promoting to 100%.

3. **If your canary deployment shows higher accuracy but also higher latency, what do you do?**  
   Do not promote automatically. Decide based on your SLOs and business priorities: if the latency exceeds your p99 SLO, roll back and investigate the latency issue separately (batching, model compression, hardware upgrade). If the latency is within SLO but higher than before, weigh the accuracy gain against the latency cost for your specific use case.